# Persian Cat Baseline: FFT Mean + CLIP

Notebook ini menjalankan baseline awal klasifikasi `real vs AI` pada subset `BigGAN / Persian cat` dengan fitur `FFT mean + CLIP`. Fokus notebook ini adalah debug data, visualisasi sederhana, ekstraksi fitur, penggabungan vektor, dan training classifier baseline.

In [ ]:
from pathlib import Path
import sys
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path("..")
MANIFEST = PROJECT_ROOT / "data" / "genimage_manifest_persian_cat.csv"
ARTIFACTS = PROJECT_ROOT / "artifacts"
PYTHON = sys.executable

FFT_OUT = ARTIFACTS / "features_fft_mean_persian_cat.csv"
CLIP_OUT = ARTIFACTS / "features_clip_persian_cat.csv"
VECTOR_OUT = ARTIFACTS / "feature_vector_classification_persian_cat.csv"
METRICS_OUT = ARTIFACTS / "results_classification_persian_cat.csv"
PRED_OUT = ARTIFACTS / "predictions_classification_persian_cat.csv"

print("Python:", PYTHON)
print("Manifest:", MANIFEST.resolve())
print("Artifacts dir:", ARTIFACTS.resolve())

## 1. Cek manifest subset

In [ ]:
df = pd.read_csv(MANIFEST)
print(df.shape)
display(df.head())
display(df[["split", "class_name", "y_ai"]].value_counts().rename("count").reset_index())

## 2. Visualisasi sampel gambar
Ambil beberapa contoh dari `ai` dan `nature` untuk memastikan subset yang dipakai memang benar.

In [ ]:
def show_grid(paths, title, ncols=4, figsize=(14, 7)):
    n = len(paths)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax, path in zip(axes, paths):
        img = Image.open(path).convert("RGB")
        ax.imshow(img)
        ax.set_title(Path(path).name, fontsize=8)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

ai_paths = df[df["y_ai"] == 1]["path"].head(8).tolist()
real_paths = df[df["y_ai"] == 0]["path"].head(8).tolist()
show_grid(ai_paths, "AI samples")
show_grid(real_paths, "Nature samples")

## 3. Demo FFT pada satu gambar AI dan satu gambar nature
Kita hitung FFT 2D, lalu lihat citra asli, `log magnitude spectrum`, dan `phase spectrum`.

In [ ]:
def load_gray(path):
    return np.asarray(Image.open(path).convert("L"), dtype=np.float32)

def fft_maps(gray):
    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    mag = np.log1p(np.abs(fshift))
    phase = np.angle(fshift)
    feats = {
        "fft_mag_mean": float(np.mean(np.abs(f))),
        "fft_phase_mean": float(np.mean(np.angle(f))),
        "fft_phase_cos_mean": float(np.mean(np.cos(np.angle(f)))),
        "fft_phase_sin_mean": float(np.mean(np.sin(np.angle(f)))),
    }
    return mag, phase, feats

def show_fft_demo(path, title):
    gray = load_gray(path)
    mag, phase, feats = fft_maps(gray)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(gray, cmap="gray")
    axes[0].set_title(f"{title} - Gray")
    axes[1].imshow(mag, cmap="magma")
    axes[1].set_title("Log Magnitude")
    axes[2].imshow(phase, cmap="twilight")
    axes[2].set_title("Phase")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return feats

ai_demo = df[df["y_ai"] == 1].iloc[0]["path"]
real_demo = df[df["y_ai"] == 0].iloc[0]["path"]

ai_feats = show_fft_demo(ai_demo, "AI")
real_feats = show_fft_demo(real_demo, "Nature")
pd.DataFrame([ai_feats, real_feats], index=["ai_demo", "nature_demo"])

## 4. Jalankan ekstraksi fitur FFT mean
Notebook ini memanggil script modular yang sudah ada, tetapi output-nya dibuat khusus untuk subset `Persian cat`.

In [ ]:
cmd = [
    PYTHON,
    str(PROJECT_ROOT / "src" / "pipeline_modular" / "features_fft_mean.py"),
    "--manifest", str(MANIFEST),
    "--output", str(FFT_OUT),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
fft_df = pd.read_csv(FFT_OUT)
print(fft_df.shape)
display(fft_df.head())
merged_fft = df[["image_id", "y_ai", "class_name"]].merge(fft_df, on="image_id", how="inner")
merged_fft.groupby("y_ai")[["fft_mag_mean", "fft_phase_mean", "fft_phase_cos_mean", "fft_phase_sin_mean"]].mean()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, color in [(1, "tab:red"), (0, "tab:blue")]:
    part = merged_fft[merged_fft["y_ai"] == label]
    axes[0].hist(part["fft_mag_mean"], bins=20, alpha=0.6, label=f"y_ai={label}", color=color)
    axes[1].hist(part["fft_phase_cos_mean"], bins=20, alpha=0.6, label=f"y_ai={label}", color=color)
axes[0].set_title("Histogram fft_mag_mean")
axes[1].set_title("Histogram fft_phase_cos_mean")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Jalankan ekstraksi fitur CLIP
CLIP dipakai sebagai image embedding. Jika notebook dijalankan pada environment yang punya dependency lengkap, mode `auto` akan mengambil embedding nyata.

In [ ]:
cmd = [
    PYTHON,
    str(PROJECT_ROOT / "src" / "pipeline_modular" / "features_clip_real.py"),
    "--manifest", str(MANIFEST),
    "--output", str(CLIP_OUT),
    "--mode", "auto",
    "--out-dim", "64",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
clip_df = pd.read_csv(CLIP_OUT)
print(clip_df.shape)
display(clip_df.head())
if "clip_source" in clip_df.columns:
    display(clip_df["clip_source"].value_counts())

## 6. Gabungkan `FFT mean + CLIP` menjadi feature vector
Di tahap ini kita sengaja tidak memasukkan NR-IQA dan periodic dulu.

In [ ]:
cmd = [
    PYTHON,
    str(PROJECT_ROOT / "src" / "pipeline_modular" / "build_feature_vector.py"),
    "--manifest", str(MANIFEST),
    "--fft", str(FFT_OUT),
    "--clip", str(CLIP_OUT),
    "--output", str(VECTOR_OUT),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
vec_df = pd.read_csv(VECTOR_OUT)
print(vec_df.shape)
display(vec_df.head())
meta_cols = ["image_id", "path", "relative_path", "generator", "split", "class_name", "is_real", "y_ai"]
feat_cols = [c for c in vec_df.columns if c not in meta_cols]
print("Feature dimension:", len(feat_cols))
print("First 10 feature cols:", feat_cols[:10])

## 7. Train classifier baseline
Karena subset ini hanya punya `train`, script akan otomatis membuat holdout split acak 80/20 untuk evaluasi awal.

In [ ]:
cmd = [
    PYTHON,
    str(PROJECT_ROOT / "src" / "pipeline_modular" / "train_classifiers.py"),
    "--features", str(VECTOR_OUT),
    "--metrics-out", str(METRICS_OUT),
    "--pred-out", str(PRED_OUT),
    "--eval-split", "val",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
metrics_df = pd.read_csv(METRICS_OUT)
display(metrics_df)

## 8. Lihat prediksi model terbaik

In [ ]:
pred_df = pd.read_csv(PRED_OUT)
best_model = metrics_df.iloc[0]["model"]
print("Best model:", best_model)
best_pred = pred_df[pred_df["model"] == best_model].copy()
display(best_pred.head())

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(best_pred["y_ai"], best_pred["pred_ai"])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["nature", "ai"]).plot(cmap="Blues")
plt.show()

## 9. Ringkasan
Jika notebook ini sudah jalan sampai akhir, baseline pertama proyek sudah valid:
1. subset data terbaca benar,
2. fitur `FFT mean` berhasil diekstrak,
3. embedding `CLIP` berhasil diekstrak,
4. vektor fitur gabungan berhasil dilatih pada task `AI vs nature`.

Langkah berikut setelah ini adalah memperluas ke kelas lain atau menambahkan ablation `NR-IQA` dan `periodic`.